In [ ]:
import json
import os
import copy
from datetime import datetime

import dataiku

In [ ]:
# Replace with the managed folder ID where exports should be written.
EXPORT_FOLDER_ID = "5fuFDP0G"

# https://developer.dataiku.com/latest/api-reference/python/projects.html#dataikuapi.dss.project.DSSProject.export_to_file
EXPORT_OPTIONS = {
    'exportUploads': True,
    'exportGitRepository': True,
    'exportAnalysisModels': True,
    'exportSavedModels': True,
    'exportModelEvaluationStores': True,
    'exportAllInputDatasets': True,
    'exportInsightsData': True,
    'exportPromptStudioHistories': True,
}

In [ ]:
export_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

client = dataiku.api_client()
export_folder_handle = dataiku.Folder(EXPORT_FOLDER_ID)

In [ ]:
# These fields should not be exported in clear text. If a key contains any of these substrings (case-insensitive), its value will be redacted.
SENSITIVE_FIELDS = [
    "password", "passwords", "apiKey", "apiKeys", "token", "tokens",
    "secret", "secrets", "privateKey", "privateKeys", "key", "appSecret",
    "appSecretContent", "credential", "credentials", "accessKey",
    "secretKey", "authToken", "connectionString", "pwd"
]

def sanitize_dict(obj):
    """Recursively redact values whose keys match known sensitive field names."""
    if not isinstance(obj, dict):
        return obj

    result = copy.deepcopy(obj)

    for key, value in list(result.items()):
        if any(sensitive in key.lower() for sensitive in SENSITIVE_FIELDS):
            if isinstance(value, str) and value:
                result[key] = "[REDACTED]"
            elif isinstance(value, dict):
                result[key] = sanitize_dict(value)
        elif isinstance(value, dict):
            result[key] = sanitize_dict(value)
        elif isinstance(value, list):
            result[key] = [sanitize_dict(item) if isinstance(item, dict) else item for item in value]

    return result

In [ ]:
project_keys = client.list_project_keys()

for project_key in project_keys:
    project = client.get_project(project_key)

    project_summary = project.get_summary()
    last_modified_timestamp = project_summary.get('versionTag', {}).get('lastModifiedOn')

    if last_modified_timestamp:
        formatted_date = datetime.fromtimestamp(last_modified_timestamp / 1000).strftime('%Y-%m-%d-%H-%M-%S')
    else:
        formatted_date = 'unknown_date'

    export_file_path = f'/tmp/{project_key}.zip'

    print(f"Exporting {project_key} ...")
    project.export_to_file(export_file_path, options=EXPORT_OPTIONS)

    upload_filename = f'PROJECT__{project_key}__{formatted_date}.zip'
    with open(export_file_path, 'rb') as f:
        export_folder_handle.upload_stream(upload_filename, f)
    os.remove(export_file_path)
    print(f"  -> uploaded as {upload_filename}")

In [ ]:
plugins = client.list_plugins()

plugin_ids = [plugin['id'] for plugin in plugins]

# Path for the output text file
plugins_output_file_path = '/tmp/plugin_ids.txt'

# Write the plugin IDs to a text file
with open(plugins_output_file_path, 'w') as output_file:
    for plugin_id in plugin_ids:
        output_file.write(plugin_id + '\n')

# Upload the text file to the managed folder
with open(plugins_output_file_path, 'rb') as f:
    export_folder_handle.upload_stream(f'list_plugins_{export_timestamp}.txt', f)

# Cleaning up the temporary file
os.remove(plugins_output_file_path)

print("Plugin IDs exported to the managed folder.")

In [ ]:
try:
    # Retrieve list of all connections
    connections = client.list_connections()

    # Prepare detailed connection information for export
    connections_details = []
    for conn in connections:
        # Obtain connection details
        conn_detail = client.get_connection(conn).get_definition()

        # Add connection details to list
        sanitized_conn = sanitize_dict(conn_detail)

        # Add sanitized connection details to list
        connections_details.append(sanitized_conn)

    # Define output file name and path
    connections_output_filepath = '/tmp/connections.txt'

    # Write connection details to JSON file
    with open(connections_output_filepath, 'w') as file:
        json.dump(connections_details, file, indent=4)


    # Upload the text file to the managed folder
    with open(connections_output_filepath, 'rb') as f:
        export_folder_handle.upload_stream(f"list_connections_{export_timestamp}.json", f)
        os.remove(connections_output_filepath)

    print("Connection details successfully exported to managed folder.")

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
# Code Environment names
code_env_name_list = sorted([env['envName'] for env in client.list_code_envs()])

with open('code_env_names.txt', 'w') as f:
    f.write('\n'.join(code_env_name_list))

# Upload the text file to the managed folder
with open('code_env_names.txt', 'rb') as f:
    export_folder_handle.upload_stream(f"list_codeenvs_{export_timestamp}.txt", f)

os.remove('code_env_names.txt')


In [ ]:
def export_code_environments():
    """Export all Python code environment definitions to a JSON file in the managed folder."""
    try:
        code_envs_list = client.list_code_envs()
        code_envs_details = {"python": []}

        for env in code_envs_list:
            env_name = env.get("envName")
            try:
                env_details = client.get_code_env("python", env_name).get_definition()
                code_envs_details["python"].append(env_details)
            except Exception as exc:
                print(f"Error exporting Python environment {env_name}: {exc}")

        output_filename = "code_environments_export.json"
        with open(output_filename, 'w') as file:
            json.dump(sanitize_dict(code_envs_details), file, indent=4)

        with open(output_filename, 'rb') as f:
            export_folder_handle.upload_stream(f"list_codeenvs_{export_timestamp}.json", f)
            os.remove(output_filename)

        print("Code environment details successfully exported to managed folder.")

    except Exception as exc:
        print(f"An error occurred during export: {exc}")


export_code_environments()

In [ ]:
settings_raw = client.get_general_settings().get_raw()
settings_output_filename = 'general_settings.json'
with open(settings_output_filename, 'w') as file:
    json.dump(sanitize_dict(settings_raw), file, indent=4)

with open(settings_output_filename, 'rb') as f:
    export_folder_handle.upload_stream(f"general_settings_{export_timestamp}.json", f)
    os.remove(settings_output_filename)

print('Finished exporting settings.')
